# Wholeshare: Privileged Context Distillation with Supervised Target Grounding (PCD + SFT)
## Production QLoRA Tuning for Qwen 3.5 9B on Cloud GPUs (Tesla P100 / T4 / L4 / A100)

This notebook implements production-grade distillation for **Qwen3.5-9B** (`Qwen/Qwen3.5-9B`) on cloud GPU infrastructure.

### Theoretical Framework & Foundational Literature:
1. **PyTorch Dynamic Graph Execution**: Paszke et al. (NeurIPS 2019, [arXiv:1912.01703](https://arxiv.org/abs/1912.01703)): Tape-based dynamic automatic differentiation and CUDA memory caching.
2. **Context Distillation & Prompt Compression**: Askell et al. (Anthropic 2021, [arXiv:2112.00861](https://arxiv.org/abs/2112.00861)) & Snell et al. (2022): Distills privileged system guidelines $x_{\text{teacher}} = (x, c_{\text{privileged}})$ into an efficient student model receiving only public prompts $x$.
3. **Generalized Knowledge Distillation (GKD)**: Agarwal et al. (ICLR 2024, [arXiv:2306.13649](https://arxiv.org/abs/2306.13649)): Forward-KL distribution matching on language generation.
4. **Self-Correction via Reinforcement Training (SCoRe)**: Kumar et al. (Google DeepMind 2024, [arXiv:2409.12917](https://arxiv.org/abs/2409.12917)): Multi-turn verifiable reward regularization to prevent policy collapse.

### Core Mathematical Formulations:
1. **Combined Distillation & SFT Objective**:
   $$\mathcal{L}_{\text{total}} = (1 - \alpha) \mathcal{L}_{\text{Forward-KL}}(s, t) + \alpha \mathcal{L}_{\text{SFT}}(s, y)$$
   where $\alpha = 0.5$. The SFT Cross-Entropy term grounds the student in exact structured JSON tool calling, while Forward-KL transfers the privileged teacher's soft reasoning distribution.
2. **Numerically Safe Soft-Log Saturation**:
   $$r = \operatorname{ReLU}(\mathcal{D}_{\text{token}} - \tau)$$
   $$\widetilde{\mathcal{D}}_{\text{token}} = \mathcal{D}_{\text{token}} - r + \log(1 + r)$$
   Because $r \ge 0$ unconditionally, $\log(1 + r)$ never evaluates negative inputs, preventing gradient `NaN`s while ensuring strictly positive gradients for outlier tokens.
3. **Exact Causal Next-Token Slices**:
   Student slice: `s_hidden[0, s_p_len - 1 : s_p_len + comp_len - 1, :]`
   Teacher slice: `t_hidden[0, t_p_len - 1 : t_p_len + comp_len - 1, :]`
   Aligns predictions token-for-token across asymmetric prompt lengths ($P_t > P_s$) with zero prompt token gradient leakage.
4. **Activation Checkpointed Chunking**:
   4. **Activation Checkpointing**: Projects vocabulary ($V = 248{,}320$) in 64-token chunks wrapped in `torch.utils.checkpoint.checkpoint(..., use_reentrant=False)`. Intermediate logits buffer is bounded to ~62 MB per tensor.
3. **VRAM Optimization**:
   Keeps the frozen embedding and output head in `COMPUTE_DTYPE` throughout setup to avoid a transient multi-gigabyte float32 cast.


## 1. Hardware Verification & Pre-Ampere Architecture Invariants


In [ ]:
import sys
import os
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError("CUDA GPU is required for training.")

device = torch.device("cuda:0")
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"GPU: {gpu_name} (Compute: sm_{capability[0]}{capability[1]})")
print(f"Dedicated VRAM: {total_vram_gb:.2f} GB")

arch_list = torch.cuda.get_arch_list()
print(f"Supported CUDA Architectures: {arch_list}")

if capability == (6, 0):
    assert any(arch in arch_list for arch in ["sm_60", "compute_60"]), (
        f"CUDA architecture sm_60 missing from PyTorch arch list: {arch_list}"
    )

# Pre-Ampere Hardware Invariants (Pascal sm_60, Turing sm_75)
if capability[0] < 8:
    print(f"\n[PRE-AMPERE GPU DETECTED (sm_{capability[0]}{capability[1]})]")
    print("- Enforcing torch.float16 (hardware lacks native BF16 tensor cores).")
    print("- Enabling Memory-Efficient SDPA; disabling FlashAttention-2.")
    COMPUTE_DTYPE = torch.float16
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
else:
    print(f"\n[AMPERE / HOPPER GPU DETECTED (sm_{capability[0]}{capability[1]})]")
    COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Selected Compute Dtype: {COMPUTE_DTYPE}")


## 2. Pinned Dependencies Installation & CUDA Binary Verification


In [ ]:
!pip install -q "transformers>=5.0.0" "peft>=0.14.0" "bitsandbytes>=0.50.2" "accelerate>=1.4.0" "datasets>=3.3.2"
!python -m bitsandbytes


## 3. Checkpointed Forward-KL + SFT Cross-Entropy Kernel

Implements the combined objective:
$$\mathcal{L} = (1 - \alpha) \sum_{t} \widetilde{\mathcal{D}}_{\text{KL}}(p_T(y_t \mid x_T) \parallel p_S(y_t \mid x_S)) + \alpha \sum_{t} \text{CrossEntropy}(p_S(\cdot \mid x_S), y_t)$$

Key Architectural Invariants:
1. **Explicit Autocast Bypass**: Performs the projection in the frozen output-head dtype, then promotes logits to float32 for stable softmax and cross-entropy calculations.
2. **Safe Soft-Log Saturation**: Uses $r = \operatorname{ReLU}(\mathcal{D} - \tau)$ and $\mathcal{D} - r + \log(1 + r)$, guaranteeing strictly positive gradients without negative argument `NaN` risks.
3. **Activation Checkpointing**: Binds intermediate logits to ~62 MB per 64-token chunk for Qwen 3.5 9B vocabulary ($V=248{,}320$). Returns `clip_count` directly to eliminate redundant kernel passes.


In [ ]:
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

def _chunk_distill_loss(
    s_h_chunk: torch.Tensor,
    t_h_chunk: torch.Tensor,
    targets_chunk: torch.Tensor,
    lm_head_weight: torch.Tensor,
    alpha: float = 0.5,
    clip_tau: float = 4.0
) -> tuple[torch.Tensor, torch.Tensor]:
    with torch.amp.autocast("cuda", enabled=False):
        projection_dtype = lm_head_weight.dtype
        s_logits = F.linear(s_h_chunk.to(projection_dtype), lm_head_weight).float()
        with torch.no_grad():
            t_logits = F.linear(t_h_chunk.to(projection_dtype), lm_head_weight).float()

        p_t = F.softmax(t_logits, dim=-1)
        log_p_t = F.log_softmax(t_logits, dim=-1)
        log_p_s = F.log_softmax(s_logits, dim=-1)

        # 1. Forward-KL Divergence summed over vocabulary
        token_kl = torch.sum(p_t * (log_p_t - log_p_s), dim=-1)

        # 2. Safe Soft-Log Clipping: r = relu(token_kl - clip_tau)
        r = F.relu(token_kl - clip_tau)
        clipped_token_kl = token_kl - r + torch.log1p(r)
        clip_count = (token_kl > clip_tau).sum().detach()

        # 3. Supervised Target Cross-Entropy Loss
        ce_loss = F.cross_entropy(s_logits, targets_chunk, reduction="sum")

        # 4. Combined Loss
        combined_loss = (1.0 - alpha) * clipped_token_kl.sum() + alpha * ce_loss
        return combined_loss, clip_count

def compute_checkpointed_distill_loss(
    s_hidden: torch.Tensor,
    t_hidden: torch.Tensor,
    target_ids: torch.Tensor,
    lm_head_weight: torch.Tensor,
    chunk_size: int = 128,
    alpha: float = 0.5,
    clip_tau: float = 4.0
) -> tuple[torch.Tensor, int]:
    comp_len = s_hidden.shape[0]
    total_loss = torch.tensor(0.0, device=s_hidden.device, dtype=torch.float32)
    total_clipped = torch.tensor(0, device=s_hidden.device, dtype=torch.long)

    for i in range(0, comp_len, chunk_size):
        end_idx = min(i + chunk_size, comp_len)
        s_c = s_hidden[i:end_idx]
        t_c = t_hidden[i:end_idx]
        y_c = target_ids[i:end_idx]

        chunk_loss, chunk_clips = checkpoint(
            _chunk_distill_loss,
            s_c,
            t_c,
            y_c,
            lm_head_weight,
            alpha,
            clip_tau,
            use_reentrant=False
        )
        total_loss = total_loss + chunk_loss
        total_clipped = total_clipped + chunk_clips

    normalized_loss = total_loss / max(1, comp_len)
    return normalized_loss, total_clipped.item()

# ------------------------------------------------------------------------------
# RIGOROUS 3-PART KERNEL SELF-TEST
# ------------------------------------------------------------------------------

# Test 1: Ground-truth unchunked vs checkpointed chunked comparison (V=64, H=128, N=16)
print("=== Running Kernel Self-Test 1: Unchunked Reference Parity ===")
t1_V, t1_H, t1_N, t1_chunk = 64, 128, 16, 4
t1_w = torch.randn(t1_V, t1_H, device="cuda", dtype=torch.float32)
t1_s_h = torch.randn(t1_N, t1_H, device="cuda", dtype=torch.float32, requires_grad=True)
t1_t_h = torch.randn(t1_N, t1_H, device="cuda", dtype=torch.float32)
t1_targets = torch.randint(0, t1_V, (t1_N,), device="cuda", dtype=torch.long)

loss_chk, clips_chk = compute_checkpointed_distill_loss(
    t1_s_h, t1_t_h, t1_targets, t1_w, chunk_size=t1_chunk, alpha=0.5, clip_tau=4.0
)
loss_chk.backward()
grad_chk = t1_s_h.grad.clone()

# Reference unchunked computation
t1_s_h_ref = t1_s_h.detach().clone().requires_grad_(True)
with torch.amp.autocast("cuda", enabled=False):
    s_l = F.linear(t1_s_h_ref, t1_w)
    with torch.no_grad():
        t_l = F.linear(t1_t_h, t1_w)
    p_t = F.softmax(t_l, dim=-1)
    log_p_t = F.log_softmax(t_l, dim=-1)
    log_p_s = F.log_softmax(s_l, dim=-1)
    kl = torch.sum(p_t * (log_p_t - log_p_s), dim=-1)
    r = F.relu(kl - 4.0)
    clipped_kl = kl - r + torch.log1p(r)
    ce = F.cross_entropy(s_l, t1_targets, reduction="sum")
    loss_ref = ((1.0 - 0.5) * clipped_kl.sum() + 0.5 * ce) / t1_N

loss_ref.backward()
grad_ref = t1_s_h_ref.grad

assert torch.allclose(loss_chk, loss_ref, atol=1e-5), f"Loss mismatch: {loss_chk.item()} vs {loss_ref.item()}"
assert torch.allclose(grad_chk, grad_ref, atol=1e-5), f"Grad mismatch: {(grad_chk - grad_ref).abs().max().item()}"
print(f"[TEST 1 PASSED] Checkpointed loss and gradient match unchunked reference exactly.")
del t1_w, t1_s_h, t1_t_h, t1_targets, t1_s_h_ref, loss_chk, loss_ref

# Test 2: Scaled logits on real Qwen 3.5 9B dimensions (H=4096, V=248320)
print("\n=== Running Kernel Self-Test 2: Soft-Log Saturation & Gradient Flow on Qwen 3.5 9B Dimensions ===")
H_dim = 4096
V_dim = 248320
test_comp_len = 16

# Scale student inputs by 5.0 to guarantee token_kl > clip_tau
test_s_h = (torch.randn(test_comp_len, H_dim, device="cuda", dtype=COMPUTE_DTYPE) * 5.0).requires_grad_(True)
test_t_h = torch.randn(test_comp_len, H_dim, device="cuda", dtype=COMPUTE_DTYPE)
test_targets = torch.randint(0, V_dim, (test_comp_len,), device="cuda", dtype=torch.long)
test_w = torch.randn(V_dim, H_dim, device="cuda", dtype=COMPUTE_DTYPE) * (1.0 / (H_dim ** 0.5))

loss_test, test_clips = compute_checkpointed_distill_loss(
    test_s_h, test_t_h, test_targets, test_w, chunk_size=8, alpha=0.5, clip_tau=4.0
)
loss_test.backward()

test_grad_norm = test_s_h.grad.norm().item()
print(f"[TEST 2 VERIFIED] Combined Distill Loss: {loss_test.item():.4f}")
print(f"[TEST 2 VERIFIED] Clipped Tokens Triggered: {test_clips}/{test_comp_len}")
print(f"[TEST 2 VERIFIED] Input Grad Norm: {test_grad_norm:.6f}")
print(f"[TEST 2 VERIFIED] Peak VRAM: {torch.cuda.max_memory_allocated() / (1024**2):.2f} MB")

assert test_clips > 0, f"Clipping should have triggered on scaled inputs, but got {test_clips}."
assert test_grad_norm > 1e-5, f"Gradient norm is zero ({test_grad_norm})! Backpropagation failed."
assert not torch.isnan(test_s_h.grad).any(), "NaN detected in gradients!"

# Release the large self-test output head before model loading.
del test_w, test_s_h, test_t_h, test_targets, loss_test
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("Self-test VRAM released cleanly.")


## 4. Deterministic FinTech Policy Verifier

Enforces core financial invariants:
1. **Integer Minor Units**: Currency amounts must be positive integer cents. Rejects floats and booleans.
2. **Soft Authorization Mandate**: PayPal escrow commitments must use Orders v2 `intent: "AUTHORIZE"`.
3. **Step-Up Verification Barrier**: Transactions $> \$1,000.00$ mandate `generate_step_up_invoice` unless `hasStepUpInvoice=True`.
4. **48-Hour Dispute Window & Financial Conservation**: Claims must be filed within 48h. Refund and disbursement must match expected pro-rata damage amounts and exactly conserve total allocation ($P_{\text{refund}} + P_{\text{disbursement}} = P_{\text{total}}$).


In [ ]:
class FinTechPolicyVerifier:
    STEP_UP_THRESHOLD_CENTS = 100000 # $1,000.00 USD

    @classmethod
    def verify_action(cls, tool_name: str, args: dict, context: dict = None) -> tuple[bool, str]:
        context = context or {}

        # 1. Enforce strict integer minor units for all monetary fields
        for k, v in args.items():
            if any(term in k.lower() for term in ["cents", "amount", "refund", "disbursement", "allocation"]):
                if isinstance(v, bool) or not isinstance(v, int):
                    return False, f"Type violation: {k}={v} (type {type(v).__name__}). Currency amounts must be strict integer minor units (cents)."
                if v < 0:
                    return False, f"Negative amount violation: {k}={v}. Currency amounts cannot be negative."

        # 2. PayPal Escrow Soft Authorization checks
        if tool_name == "authorize_paypal_escrow":
            intent = args.get("intent", "AUTHORIZE")
            if intent != "AUTHORIZE":
                return False, f"Prohibited intent: {intent}. Orders v2 soft hold requires intent: AUTHORIZE."

            if "amountCents" not in args:
                return False, "Missing required argument: amountCents."
            amount = args["amountCents"]

            exp_amount = context.get("expected_amount_cents")
            if exp_amount is not None and amount != exp_amount:
                return False, f"Amount mismatch: got {amount} cents, expected {exp_amount} cents from prompt."

            has_step_up = context.get("hasStepUpInvoice", False)
            if amount > cls.STEP_UP_THRESHOLD_CENTS and not has_step_up:
                return False, f"Amount {amount} cents exceeds $1,000 step-up threshold without prior step-up invoice."

        # 3. Step-Up KYC Invoice checks
        elif tool_name == "generate_step_up_invoice":
            if "amountCents" not in args:
                return False, "Missing required argument: amountCents."
            amount = args["amountCents"]

            exp_amount = context.get("expected_amount_cents")
            if exp_amount is not None and amount != exp_amount:
                return False, f"Amount mismatch: got {amount} cents, expected {exp_amount} cents from prompt."

            if amount <= cls.STEP_UP_THRESHOLD_CENTS:
                return False, f"Step-up invoice requested for {amount} cents, but amount is <= $1,000 threshold."

        # 4. Dispute Timeliness & Conservation checks
        elif tool_name == "resolve_dispute":
            elapsed_hours = context.get("elapsed_hours", 24)
            decision = args.get("decision", "")

            refund = args.get("refundAmountCents", args.get("refundCents"))
            release = args.get("supplierReleaseCents", args.get("disbursementCents"))
            if isinstance(refund, bool) or not isinstance(refund, int) or refund < 0:
                return False, "Dispute refund must be a non-negative integer number of cents."
            if isinstance(release, bool) or not isinstance(release, int) or release < 0:
                return False, "Supplier release must be a non-negative integer number of cents."

            if elapsed_hours > 48:
                if decision != "REJECTED_OUTSIDE_48HR_WINDOW":
                    return False, f"Delivery was {elapsed_hours}h ago (> 48h limit). Decision must be REJECTED_OUTSIDE_48HR_WINDOW (got {decision})."
                if refund != 0:
                    return False, f"Expired dispute cannot issue refund ({refund} cents)."
            else:
                if decision not in ("PARTIAL_REFUND_APPROVED", "APPROVED_PRO_RATA_REFUND"):
                    return False, f"Timely dispute ({elapsed_hours}h <= 48h) must use an approved pro-rata decision (got {decision})."

                total_alloc = context.get("total_allocation_cents")
                if total_alloc is not None and refund + release != total_alloc:
                    return False, f"Conservation violation: refund ({refund}) + release ({release}) != total allocation ({total_alloc})."

                exp_refund = context.get("expected_refund_cents")
                if exp_refund is not None and refund != exp_refund:
                    return False, f"Split violation: refund {refund} cents != expected {exp_refund} cents."

                exp_release = context.get("expected_disbursement_cents")
                if exp_release is not None and release != exp_release:
                    return False, f"Split violation: supplier release {release} cents != expected {exp_release} cents."

        return True, "VALID"

print("FinTechPolicyVerifier registered.")


## 5. Dataset Ingestion & Curriculum Verification

Mounts the authentic B2B commerce curriculum dataset (`opsd_train_curriculum.jsonl`) from `/kaggle/input`.
Validates the presence of `student_prompt`, `teacher_prompt`, and `target_completion` across all records.


In [ ]:
import json
import glob
from datasets import Dataset

LOCAL_PATH = "data/lake/opsd_preprocessed/opsd_train_curriculum.jsonl"
LOCAL_VAL_PATH = "data/lake/opsd_preprocessed/opsd_val_curriculum.jsonl"
LOCAL_CURATED_PATH = "data/lake/opsd_training_curated.jsonl"
IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    candidates = glob.glob("/kaggle/input/**/opsd_train_curriculum.jsonl", recursive=True)
    if candidates:
        DATASET_FILE = candidates[0]
        data_dir = os.path.dirname(DATASET_FILE)
        val_cand = os.path.join(data_dir, "opsd_val_curriculum.jsonl")
        VAL_DATASET_FILE = val_cand if os.path.exists(val_cand) else None
        CURATED_DATASET_FILE = os.path.join(data_dir, "opsd_training_curated.jsonl")
        if not os.path.exists(CURATED_DATASET_FILE):
            CURATED_DATASET_FILE = None
        print(f"[KAGGLE DATASET FOUND] Located training curriculum at: {DATASET_FILE}")
        if VAL_DATASET_FILE:
            print(f"[KAGGLE DATASET FOUND] Located validation curriculum at: {VAL_DATASET_FILE}")
    else:
        available_dirs = glob.glob("/kaggle/input/*")
        raise FileNotFoundError(
            f"[FATAL CONFIGURATION ERROR] Running on Kaggle but opsd_train_curriculum.jsonl not found in /kaggle/input!\n"
            f"Available directories: {available_dirs}"
        )
elif os.path.exists(LOCAL_PATH):
    DATASET_FILE = LOCAL_PATH
    VAL_DATASET_FILE = LOCAL_VAL_PATH if os.path.exists(LOCAL_VAL_PATH) else None
    CURATED_DATASET_FILE = LOCAL_CURATED_PATH if os.path.exists(LOCAL_CURATED_PATH) else None
else:
    raise FileNotFoundError(f"[FATAL] Dataset file not found at {LOCAL_PATH}.")

def load_opsd_curriculum_data(filepath: str) -> Dataset:
    records = []
    required_keys = {"student_prompt", "teacher_prompt", "target_completion"}
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            item = json.loads(line)
            if not required_keys.issubset(item.keys()):
                raise ValueError(f"Line {line_num} in {filepath} missing required keys: {required_keys - set(item.keys())}")
            records.append(item)
    print(f"Loaded and verified {len(records)} authentic curriculum records from {filepath}.")
    return Dataset.from_list(records)

raw_train_data = load_opsd_curriculum_data(DATASET_FILE)
if CURATED_DATASET_FILE:
    curated_data = load_opsd_curriculum_data(CURATED_DATASET_FILE)
    combined = list(raw_train_data) + list(curated_data)
    seen_ids = set()
    unique_records = []
    for record in combined:
        record_id = record.get("id")
        if record_id is None or record_id not in seen_ids:
            unique_records.append(record)
            if record_id is not None:
                seen_ids.add(record_id)
    raw_train_data = Dataset.from_list(unique_records)
print(f"Active training curriculum: {len(raw_train_data)} records.")


## 6. Base Model (Qwen 3.5 9B), QLoRA Setup & embed_tokens VRAM Optimization

Key Optimizations:
1. Targets **`Qwen/Qwen3.5-9B`** with exact hidden size $H=4096$ and vocabulary size $V=248{,}320$.
2. Single-device placement `device_map={"": 0}` on GPU 0.
3. Keeps frozen non-quantized modules in `COMPUTE_DTYPE` during setup to avoid PEFT's transient multi-gigabyte float32 cast.
4. Targets LoRA across all projection modules (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj, in_proj_qkv, out_proj`).
5. Keeps the frozen output head in the selected compute dtype and promotes chunk logits to float32 before softmax and cross-entropy.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen3.5-9B"

print(f"Loading {MODEL_ID} in 4-bit NormalFloat4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
    trust_remote_code=True
)

for parameter in model.parameters():
    parameter.requires_grad_(False)
if not hasattr(model, "gradient_checkpointing_enable"):
    raise RuntimeError("Loaded model does not support gradient checkpointing.")
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    raise RuntimeError("Loaded model cannot enable input gradients required for checkpointed LoRA training.")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "in_proj_qkv", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Keep the frozen embedding table in the selected compute dtype.
base_backbone = model.get_base_model() if hasattr(model, "get_base_model") else model
if hasattr(base_backbone, "model") and hasattr(base_backbone.model, "language_model"):
    embed_module = base_backbone.model.language_model.embed_tokens
elif hasattr(base_backbone, "model") and hasattr(base_backbone.model, "embed_tokens"):
    embed_module = base_backbone.model.embed_tokens
else:
    embed_module = base_backbone.embed_tokens

embed_module.to(COMPUTE_DTYPE)

raw_head = getattr(model, "lm_head", None)
if raw_head is not None and hasattr(raw_head, "weight"):
    lm_head_weight = raw_head.weight
else:
    lm_head_weight = model.get_base_model().lm_head.weight

assert lm_head_weight.dtype == COMPUTE_DTYPE, f"lm_head_weight must match compute dtype {COMPUTE_DTYPE}, got {lm_head_weight.dtype}"

print(f"Model loaded and optimized. VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")


## 7. Asymmetric Span Collator with Official ChatML Templating & Oversize Filtering

Key Invariants:
1. Uses `apply_chat_template(..., tokenize=False)` followed by `tokenizer.encode()` to guarantee pure `list[int]`.
2. Preserves structural prompt tags (`<|im_start|>assistant\n`) and completion closing tokens (`<|im_end|>`).
3. Never slices prompt tails or completion syntax. Samples exceeding `max_length` are dropped cleanly and logged to audit skip rates.


In [ ]:
class AsymmetricSpanCollator:
    def __init__(self, tokenizer, max_length: int = 2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.total_samples = 0
        self.skipped_samples = 0

    def __call__(self, batch: list[dict]) -> dict | None:
        item = batch[0] # Micro-batch size 1
        self.total_samples += 1

        s_messages = [
            {"role": "system", "content": "You are Wholeshare's autonomous procurement agent. Evaluate anti-fraud policy constraints and emit compliant structured JSON tool calls."},
            {"role": "user", "content": item["student_prompt"]}
        ]
        t_messages = [
            {"role": "system", "content": "You are Wholeshare's autonomous procurement agent. Evaluate anti-fraud policy constraints and emit compliant structured JSON tool calls."},
            {"role": "user", "content": item["teacher_prompt"]}
        ]

        s_p_text = self.tokenizer.apply_chat_template(s_messages, tokenize=False, add_generation_prompt=True)
        t_p_text = self.tokenizer.apply_chat_template(t_messages, tokenize=False, add_generation_prompt=True)

        s_p_ids = self.tokenizer.encode(s_p_text, add_special_tokens=False)
        t_p_ids = self.tokenizer.encode(t_p_text, add_special_tokens=False)

        comp_text = item["target_completion"]
        if not comp_text.endswith("<|im_end|>"):
            comp_text = comp_text + "<|im_end|>"
        comp_ids = self.tokenizer.encode(comp_text, add_special_tokens=False)

        s_full = s_p_ids + comp_ids
        t_full = t_p_ids + comp_ids

        # Never slice prompt generation tails or completion JSON tokens.
        # Cleanly skip sequences exceeding max_length to preserve exact syntax.
        if len(s_full) > self.max_length or len(t_full) > self.max_length:
            self.skipped_samples += 1
            return None

        return {
            "s_input_ids": torch.tensor([s_full], dtype=torch.long),
            "s_attention_mask": torch.ones(1, len(s_full), dtype=torch.long),
            "t_input_ids": torch.tensor([t_full], dtype=torch.long),
            "t_attention_mask": torch.ones(1, len(t_full), dtype=torch.long),
            "target_ids": torch.tensor(comp_ids, dtype=torch.long),
            "s_prompt_len": len(s_p_ids),
            "t_prompt_len": len(t_p_ids),
            "comp_len": len(comp_ids)
        }

collator = AsymmetricSpanCollator(tokenizer, max_length=2048)
sample_b = collator([raw_train_data[0]])
if sample_b is not None:
    print(f"Sample Student tokens: {sample_b['s_input_ids'].shape[1]} (Prompt: {sample_b['s_prompt_len']}, Comp: {sample_b['comp_len']})")
    print(f"Sample Teacher tokens: {sample_b['t_input_ids'].shape[1]} (Prompt: {sample_b['t_prompt_len']}, Comp: {sample_b['comp_len']})")


## 8. Baseline Anti-Catastrophic Forgetting & Pre-Training FinTech Gauntlet

Evaluates:
1. **Pre-Training Anti-Catastrophic Forgetting Benchmark (N=25)**: Evaluates held-out financial math reasoning problems before fine-tuning.
   - Extracts numbers matching `Answer: <number>`.
   - Compares with relative and absolute tolerances, accepting standard percentage notations.
   - Computes true 95% Wilson score confidence intervals reported as `[lower, upper]`.
2. **Pre-Training FinTech Policy Gauntlet (N=5)**: Evaluates the unmodified base model on the 5 policy scenarios using the authentic training system prompt, establishing a rigorous pre-tuning baseline.


In [ ]:
import re
import math

anchor_eval_suite = [
    {"q": "Operating revenue is $850,000 and operating expenses are $680,000. Calculate operating income.", "expected": 170000.0},
    {"q": "Total liabilities are $4,200 million and total assets are $10,500 million. Calculate debt-to-assets percentage.", "expected": 40.0},
    {"q": "Gross profit is $320,000 on net sales of $800,000. Calculate gross profit margin percentage.", "expected": 40.0},
    {"q": "Cash flow from operations is $450,000, capital expenditures are $120,000. Calculate free cash flow.", "expected": 330000.0},
    {"q": "Current assets are $600,000 and current liabilities are $240,000. Calculate current ratio.", "expected": 2.5},
    {"q": "Net sales are $1,200,000 and cost of goods sold is $720,000. Calculate gross profit.", "expected": 480000.0},
    {"q": "Total debt is $500,000 and total equity is $1,000,000. Calculate debt to equity ratio.", "expected": 0.5},
    {"q": "Net income is $90,000 on total assets of $900,000. Calculate Return on Assets (ROA) percentage.", "expected": 10.0},
    {"q": "EBIT is $250,000 and interest expense is $50,000. Calculate interest coverage ratio.", "expected": 5.0},
    {"q": "Credit sales are $400,000 and average accounts receivable is $80,000. Calculate accounts receivable turnover.", "expected": 5.0},
    {"q": "Operating cash flow is $600,000 and net income is $500,000. Calculate cash flow to net income ratio.", "expected": 1.2},
    {"q": "Beginning inventory is $50,000, ending inventory is $70,000, COGS is $360,000. Calculate inventory turnover.", "expected": 6.0},
    {"q": "Net profit is $150,000 on shareholder equity of $750,000. Calculate ROE percentage.", "expected": 20.0},
    {"q": "Total revenue is $2,000,000 and total operating costs are $1,700,000. Calculate operating margin percentage.", "expected": 15.0},
    {"q": "Market price per share is $60 and earnings per share is $4. Calculate P/E ratio.", "expected": 15.0},
    {"q": "Annual dividend is $3.00 and stock price is $60.00. Calculate dividend yield percentage.", "expected": 5.0},
    {"q": "Quick assets are $300,000 and current liabilities are $150,000. Calculate quick ratio.", "expected": 2.0},
    {"q": "Current assets are $500,000 and current liabilities are $350,000. Calculate working capital.", "expected": 150000.0},
    {"q": "Book value of equity is $1,200,000 with 300,000 shares outstanding. Calculate book value per share.", "expected": 4.0},
    {"q": "Enterprise value is $5,000,000 and EBITDA is $1,000,000. Calculate EV to EBITDA multiple.", "expected": 5.0},
    {"q": "Net income is $200,000 on total sales of $2,500,000. Calculate net profit margin percentage.", "expected": 8.0},
    {"q": "Cash is $80,000, marketable securities are $40,000, current liabilities are $100,000. Calculate cash ratio.", "expected": 1.2},
    {"q": "Cost of goods sold is $540,000 and average inventory is $90,000. Calculate days sales in inventory using 360 days.", "expected": 60.0},
    {"q": "Total assets are $2,000,000 and net sales are $4,000,000. Calculate asset turnover ratio.", "expected": 2.0},
    {"q": "Net credit sales are $730,000 and average receivables are $100,000. Calculate days sales outstanding using 365 days.", "expected": 50.0}
]

def extract_numerical_answer(text: str) -> float | None:
    cleaned = re.sub(r"[\$,]", "", text)
    # Search for structured Answer: <num> first
    m = re.findall(r"(?:Answer|Final Answer|Result)[\s:]*([-+]?\d+(?:\.\d+)?)", cleaned, re.IGNORECASE)
    if m:
        try:
            return float(m[-1])
        except ValueError:
            pass
    m2 = re.findall(r"(?:Total|Income|Ratio|Margin|Multiple|Turnover)[\s:]*([-+]?\d+(?:\.\d+)?)", cleaned, re.IGNORECASE)
    if m2:
        try:
            return float(m2[-1])
        except ValueError:
            pass
    m3 = re.findall(r"[-+]?\d+(?:\.\d+)?", cleaned)
    if m3:
        try:
            return float(m3[-1])
        except ValueError:
            pass
    return None

def check_numerical_match(pred: float, exp: float) -> bool:
    tol = max(0.05, 0.01 * abs(exp))
    if abs(pred - exp) <= tol:
        return True
    if abs(pred * 100.0 - exp) <= tol:
        return True
    if abs(pred - exp / 100.0) <= tol:
        return True
    return False

def compute_wilson_ci(correct: int, total: int, confidence: float = 0.95) -> tuple[float, float, float]:
    if total == 0:
        return 0.0, 0.0, 0.0
    p_hat = correct / total
    z = 1.96
    denominator = 1.0 + (z**2) / total
    center = (p_hat + (z**2) / (2 * total)) / denominator
    spread = (z / denominator) * math.sqrt((p_hat * (1.0 - p_hat) / total) + (z**2) / (4 * total**2))
    lower = max(0.0, center - spread)
    upper = min(1.0, center + spread)
    return p_hat, lower, upper

def evaluate_anchor_accuracy(eval_model, eval_tokenizer, suite):
    eval_model.eval()
    if hasattr(eval_model, "gradient_checkpointing_disable"):
        try:
            eval_model.gradient_checkpointing_disable()
        except Exception:
            pass
    correct_indices = set()
    for idx, item in enumerate(suite):
        prompt_text = item["q"] + "\nProvide your derivation, and conclude your response with 'Answer: <number>'."
        msgs = [
            {"role": "system", "content": "You are a financial analyst. Compute the exact numerical answer."},
            {"role": "user", "content": prompt_text}
        ]
        prompt = eval_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = eval_tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = eval_model.generate(**inp, max_new_tokens=350, do_sample=False, pad_token_id=eval_tokenizer.pad_token_id)
        resp = eval_tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_val = extract_numerical_answer(resp)
        exp_val = item["expected"]
        if pred_val is not None and check_numerical_match(pred_val, exp_val):
            correct_indices.add(idx)

    eval_model.train()
    if hasattr(eval_model, "gradient_checkpointing_enable"):
        try:
            eval_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except Exception:
            pass

    p_hat, lower, upper = compute_wilson_ci(len(correct_indices), len(suite))
    return p_hat, lower, upper, correct_indices

def extract_first_json_object(text: str) -> dict | None:
    start = -1
    depth = 0
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if ch == '"' and not escape:
            in_string = not in_string
        elif not in_string:
            if ch == '{':
                if depth == 0:
                    start = i
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0 and start != -1:
                    try:
                        return json.loads(text[start : i + 1])
                    except Exception:
                        start = -1
        escape = (ch == '\\' and not escape)
    return None

eval_scenarios = [
    {
        "name": "High-Value Step-Up KYC Enforcement",
        "prompt": "Buyer Austin Bakery submits an escrow commitment of $2,450.00 USD (245000 cents) for Clan clan_flour_01.\nMember Spend Policy Cap: $3,000.00 USD (300,000 cents).\nStep-Up KYC Threshold: $1,000.00 USD (100,000 cents).\nPayment Method: credit_card.\nEvaluate anti-fraud policy sandwich constraints and trigger the compliant PayPal action.",
        "expect_tool": "generate_step_up_invoice",
        "context": {"expected_amount_cents": 245000, "hasStepUpInvoice": False}
    },
    {
        "name": "High-Value Authorized Escrow Hold with Prior Step-Up",
        "prompt": "Buyer Austin Bakery has verified Step-Up KYC Invoice INV2_98712 for $2,450.00 USD (245000 cents). Proceed with Orders v2 escrow authorization for Clan clan_flour_01.\nMember Spend Policy Cap: $3,000.00 USD (300,000 cents).\nStep-Up Verification Status: COMPLETED.",
        "expect_tool": "authorize_paypal_escrow",
        "context": {"expected_amount_cents": 245000, "hasStepUpInvoice": True}
    },
    {
        "name": "Low-Value Direct Soft Hold Enforcement",
        "prompt": "Buyer GreenLeaf submits an escrow commitment of $420.00 USD (42000 cents) for Clan clan_packaging_02.\nMember Spend Policy Cap: $1,000.00 USD (100,000 cents).\nStep-Up KYC Threshold: $1,000.00 USD (100,000 cents).\nPayment Method: credit_card.\nEvaluate anti-fraud policy sandwich constraints and trigger the compliant PayPal action.",
        "expect_tool": "authorize_paypal_escrow",
        "context": {"expected_amount_cents": 42000, "hasStepUpInvoice": False}
    },
    {
        "name": "Damaged Cargo Dispute Mediation (Timely Settlement with Exact Math)",
        "prompt": "Dispute claim filed for Clan clan_delivery_0001 by Buyer sme_buyer_001.\nCarrier Tracking: FX_783492100000 (Status: DELIVERED, Elapsed: 12 hours ago).\nInspection Window Limit: 48 hours.\nTotal Member Allocation: $2,000.00 USD (200000 cents).\nDamage Reported: 20% volume affected.\nAttached Damage Photo: sha256:eea985d46a8545fe7fd3506bce1c85d8d9d1777d7cd1a8386c992172c35131c2\n\nVerify timeliness, calculate exact dispute refund (20%) and supplier disbursement (80%), and trigger dispute mediation.",
        "expect_tool": "resolve_dispute",
        "context": {"elapsed_hours": 12, "total_allocation_cents": 200000, "expected_refund_cents": 40000, "expected_disbursement_cents": 160000}
    },
    {
        "name": "Expired Damage Dispute Rejection (> 48h Window)",
        "prompt": "Dispute claim filed for Clan clan_delivery_0002 by Buyer sme_buyer_002.\nCarrier Tracking: FX_992182000000 (Status: DELIVERED, Elapsed: 72 hours ago).\nInspection Window Limit: 48 hours.\nTotal Member Allocation: $1,500.00 USD (150000 cents).\nDamage Reported: 10% volume affected.\n\nVerify timeliness and trigger dispute mediation decision.",
        "expect_tool": "resolve_dispute",
        "context": {"elapsed_hours": 72, "total_allocation_cents": 150000, "expected_refund_cents": 0}
    }
]

def run_fintech_gauntlet(eval_model, eval_tokenizer, scenarios, title="Gauntlet"):
    eval_model.eval()
    if hasattr(eval_model, "gradient_checkpointing_disable"):
        try:
            eval_model.gradient_checkpointing_disable()
        except Exception:
            pass

    eos_ids = [eval_tokenizer.eos_token_id]
    im_end_id = eval_tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id != eval_tokenizer.unk_token_id:
        eos_ids.append(im_end_id)

    training_system_prompt = "You are Wholeshare's autonomous procurement agent. Evaluate anti-fraud policy constraints and emit compliant structured JSON tool calls."
    passes = 0
    results = []
    print(f"=== Running {title} ({len(scenarios)} Scenarios) ===\n")

    for sc in scenarios:
        msgs = [
            {"role": "system", "content": training_system_prompt},
            {"role": "user", "content": sc["prompt"]}
        ]
        test_p = eval_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = eval_tokenizer(test_p, return_tensors="pt").to(device)
        with torch.no_grad():
            out = eval_model.generate(
                **inp,
                max_new_tokens=600,
                do_sample=False,
                eos_token_id=eos_ids,
                pad_token_id=eval_tokenizer.pad_token_id
            )
        resp = eval_tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        tool_data = extract_first_json_object(resp)

        valid = False
        reason = "No valid JSON tool call detected."
        t_name = ""
        if tool_data is not None:
            try:
                t_name = tool_data.get("tool") or tool_data.get("name") or tool_data.get("action", "")
                t_args = tool_data.get("arguments") or tool_data.get("details") or tool_data
                valid, reason = FinTechPolicyVerifier.verify_action(t_name, t_args, sc["context"])
                is_expected = (t_name == sc["expect_tool"])
                if not is_expected:
                    valid = False
                    reason = f"Expected tool {sc['expect_tool']}, got {t_name}."
            except Exception as e:
                valid = False
                reason = f"Verification error: {e}"

        if valid:
            passes += 1
            print(f"[{sc['name']}] PASS: {t_name}")
        else:
            print(f"[{sc['name']}] FAIL: {reason}")
        results.append({"name": sc["name"], "passed": valid, "reason": reason})

    print(f"\n{title} Score: {passes}/{len(scenarios)} Passed.")

    eval_model.train()
    if hasattr(eval_model, "gradient_checkpointing_enable"):
        try:
            eval_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except Exception:
            pass

    return passes, results

pre_acc, pre_lower, pre_upper, pre_correct_set = evaluate_anchor_accuracy(model, tokenizer, anchor_eval_suite)
print(f"Pre-Training Anchor Accuracy: {pre_acc*100:.1f}% (95% Wilson CI: [{pre_lower*100:.1f}%, {pre_upper*100:.1f}%], N={len(anchor_eval_suite)})")

pre_gauntlet_score, _ = run_fintech_gauntlet(model, tokenizer, eval_scenarios, title="Pre-Training Base Model Gauntlet")


## 9. Production Distillation Training Loop for Qwen 3.5 9B

Features:
1. **Explicit RUN_MODE Constant**: Clearly toggles between `"FULL_TRAINING"` (one pass over curriculum and curated OPSD records) and `"SMOKE_TEST"` (5 steps). Tags output directory (`_SMOKE`).
2. **Inner Backbone Extraction**: Extracts language model backbone (`model.model.language_model` or `model.model`) returning `[1, S, 4096]` hidden states.
3. **Teacher Pass under `disable_adapter()`**: Frozen reference model pass without allocating a second model copy.
4. **Causal Next-Token Slices**: Exact causal shift extraction `hidden[0, P-1 : P+C-1, :]`.
5. **Combined Forward-KL + SFT Loss**: Checkpointed chunking of size 64 with soft-log clipping and target token cross-entropy. Returns `clip_count` directly.
6. **Non-Corrupting Non-Finite Loss Guard**: Skips backward pass on bad micro-batches without wiping previously accumulated gradients or desynchronizing boundary steps.
7. **Periodic Checkpoint Saving**: Automatically serializes checkpoints every 250 steps.
8. **Mathematical Gradient Rescaling on Flush**: Accurately rescales partial epoch boundary accumulations to the true mean.


In [ ]:
import time
import csv
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
import bitsandbytes as bnb

# ==============================================================================
# RUN EXECUTION MODE CONFIGURATION
# - "FULL_TRAINING": one epoch over curriculum and curated OPSD records
# - "SMOKE_TEST": 5 optimization steps (pipeline verification)
# ==============================================================================
RUN_MODE = "SMOKE_TEST" # Switch to FULL_TRAINING only after the GPU smoke run passes and runtime is budgeted.
SMOKE_TEST = (RUN_MODE == "SMOKE_TEST")
MAX_STEPS = 5 if SMOKE_TEST else 1000000
OUTPUT_DIR = "./wholeshare_qwen35_9b_adapter_SMOKE" if SMOKE_TEST else "./wholeshare_qwen35_9b_opsd_adapter"
METRICS_CSV = f"{OUTPUT_DIR}_metrics.csv"
METRICS_JSON = f"{OUTPUT_DIR}_metrics.json"
STOP_FILE = Path(f"{OUTPUT_DIR}.STOP")

EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8 # Effective batch size 8
LEARNING_RATE = 1.5e-4 # Calibrated learning rate for LoRA adapters
MIN_LR = 1.5e-5 # Learning rate decay floor
ALPHA_SFT = 0.5 # 50% Forward-KL soft distillation, 50% target token cross-entropy

if SMOKE_TEST:
    print("!" * 75)
    print("!  WARNING: SMOKE_TEST MODE ACTIVE (5 STEPS ONLY)                           !")
    print(f"!  Exporting to tagged directory: {OUTPUT_DIR:<42} !")
    print("!  Results are for pipeline verification only, NOT multi-epoch convergence. !")
    print("!" * 75)
else:
    print("=" * 75)
    print(f"  FULL TRAINING MODE ACTIVE: {MAX_STEPS} steps targeting Wholeshare commerce curriculum.")
    print("=" * 75)

dataloader = DataLoader(raw_train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)

trainable_params = [p for p in model.parameters() if p.requires_grad]
try:
    optimizer = bnb.optim.PagedAdamW8bit(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)
    print("Using bnb.optim.PagedAdamW8bit optimizer.")
except Exception as e:
    print(f"PagedAdamW8bit fallback to AdamW: {e}")
    optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)

total_training_steps = min(MAX_STEPS, (len(dataloader) * EPOCHS + GRAD_ACCUM_STEPS - 1) // GRAD_ACCUM_STEPS)
print(f"Planned one-pass sample coverage: {min(len(dataloader), total_training_steps * GRAD_ACCUM_STEPS)}/{len(dataloader)} records before over-length skips.")
warmup_steps = min(50, max(2, total_training_steps // 10))
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16))

print("=" * 70)
print(f"  STARTING WHOLESHARE QWEN 3.5 9B DISTILLATION (STEPS: {total_training_steps}, BATCH: {GRAD_ACCUM_STEPS})")
print("=" * 70)

model.train()
step = 0
step_loss_acc = 0.0
step_clips_acc = 0
step_tokens_acc = 0
pending_microbatches = 0

window_loss = 0.0
window_clips = 0
window_tokens = 0
window_steps = 0

optimizer.zero_grad(set_to_none=True)

# Extract backbone dynamically for Qwen 3.5 9B
base_m = model.get_base_model() if hasattr(model, "get_base_model") else model
if hasattr(base_m, "model") and hasattr(base_m.model, "language_model"):
    backbone = base_m.model.language_model
elif hasattr(base_m, "model"):
    backbone = base_m.model
else:
    backbone = base_m

step_start_time = time.perf_counter()
run_start_time = step_start_time
batch_idx = 0
stop_requested = False
nonfinite_batches = 0
skipped_batches = 0
optimizer_updates = 0
metrics = []

for epoch in range(EPOCHS):
    for batch_idx, batch in enumerate(dataloader):
        if STOP_FILE.exists():
            print(f"[STOP REQUESTED] Found {STOP_FILE}. Flushing accumulated gradients and ending training.")
            stop_requested = True
            break
        if batch is None:
            skipped_batches += 1
            continue

        s_ids = batch["s_input_ids"].to(device)
        s_mask = batch["s_attention_mask"].to(device)
        t_ids = batch["t_input_ids"].to(device)
        t_mask = batch["t_attention_mask"].to(device)
        target_ids = batch["target_ids"].to(device)

        s_p_len = batch["s_prompt_len"]
        t_p_len = batch["t_prompt_len"]
        comp_len = batch["comp_len"]

        if comp_len == 0:
            continue

        with torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=True):
            # 1. Student Forward Pass (Active LoRA Adapter)
            s_out = backbone(input_ids=s_ids, attention_mask=s_mask, use_cache=False)
            s_hidden = s_out[0]

            # 2. Teacher Forward Pass (Disabled Adapter = Frozen Reference Model)
            with torch.no_grad():
                with model.disable_adapter():
                    t_out = backbone(input_ids=t_ids, attention_mask=t_mask, use_cache=False)
                    t_hidden = t_out[0]

            # 3. Exact Causal Shift Extraction
            s_comp_h = s_hidden[0, s_p_len - 1 : s_p_len + comp_len - 1, :]
            t_comp_h = t_hidden[0, t_p_len - 1 : t_p_len + comp_len - 1, :]

            # 4. Checkpointed Combined Distill Loss (Forward-KL + SFT Cross Entropy)
            loss, clips_count = compute_checkpointed_distill_loss(
                s_comp_h,
                t_comp_h,
                target_ids,
                lm_head_weight,
                chunk_size=64,
                alpha=ALPHA_SFT,
                clip_tau=4.0
            )
            scaled_loss = loss / GRAD_ACCUM_STEPS

        # Non-corrupting non-finite loss guard: skip micro-batch without wiping accumulated gradients
        if torch.isfinite(loss):
            scaler.scale(scaled_loss).backward()
            step_loss_acc += loss.item() / GRAD_ACCUM_STEPS
            step_clips_acc += clips_count
            step_tokens_acc += comp_len
        else:
            nonfinite_batches += 1
            print(f"[WARNING] Non-finite loss at batch {batch_idx} (step {step}): {loss.item()}. Skipping micro-batch backward pass.")

        pending_microbatches += 1

        # 6. Optimizer Step on Accumulation Boundary
        if pending_microbatches >= GRAD_ACCUM_STEPS:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending_microbatches = 0
            lr_scheduler.step()

            # Enforce learning rate floor only after warmup phase
            cur_lr = lr_scheduler.get_last_lr()[0]
            if step > warmup_steps:
                cur_lr = max(MIN_LR, cur_lr)
                for pg in optimizer.param_groups:
                    pg["lr"] = cur_lr

            step += 1
            optimizer_updates += 1
            window_loss += step_loss_acc
            window_clips += step_clips_acc
            window_tokens += step_tokens_acc
            window_steps += 1

            step_loss_acc = 0.0
            step_clips_acc = 0
            step_tokens_acc = 0

            if step % 10 == 0 or step == 1 or step == total_training_steps:
                cur_vram = torch.cuda.memory_allocated() / (1024**2)
                avg_l = window_loss / max(1, window_steps)
                clip_frac = window_clips / max(1, window_tokens)
                actual_lr = optimizer.param_groups[0]["lr"]
                elapsed_step_time = (time.perf_counter() - step_start_time) / max(1, window_steps)
                row = {"step": step, "combined_loss": avg_l, "clip_fraction": clip_frac, "seconds_per_optimizer_step": elapsed_step_time, "learning_rate": actual_lr, "allocated_vram_mb": cur_vram, "tokens_in_window": window_tokens, "nonfinite_batches": nonfinite_batches, "skipped_batches": skipped_batches}
                metrics.append(row)
                with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
                    writer = csv.DictWriter(f, fieldnames=row.keys())
                    writer.writeheader()
                    writer.writerows(metrics)
                with open(METRICS_JSON, "w", encoding="utf-8") as f:
                    json.dump({"run_mode": RUN_MODE, "model_id": MODEL_ID, "planned_steps": total_training_steps, "completed_steps": step, "optimizer_updates": optimizer_updates, "elapsed_seconds": time.perf_counter() - run_start_time, "nonfinite_batches": nonfinite_batches, "skipped_batches": skipped_batches, "metrics": metrics}, f, indent=2)
                print(f"Step {step:04d}/{total_training_steps} | Combined Loss: {avg_l:.4f} | Clip Frac: {clip_frac*100:.1f}% | Latency: {elapsed_step_time:.2f} s/step | LR: {actual_lr:.2e} | VRAM: {cur_vram:.1f} MB")
                window_loss = 0.0
                window_clips = 0
                window_tokens = 0
                window_steps = 0
                step_start_time = time.perf_counter()

            # Periodic Checkpoint Serialization every 250 steps
            if step % 250 == 0 and step < total_training_steps:
                ckpt_dir = f"{OUTPUT_DIR}_checkpoint_step_{step}"
                model.save_pretrained(ckpt_dir)
                print(f"[CHECKPOINT SAVED] Periodic checkpoint exported to {ckpt_dir}")

            if step >= total_training_steps:
                break

    # Flush remaining accumulated gradients at epoch boundary with exact mathematical rescaling
    rem_batches = pending_microbatches
    if rem_batches != 0 and step < total_training_steps:
        rescale_factor = GRAD_ACCUM_STEPS / rem_batches
        for p in trainable_params:
            if p.grad is not None:
                p.grad.mul_(rescale_factor)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        pending_microbatches = 0
        optimizer_updates += 1
        lr_scheduler.step()
        if step > warmup_steps:
            cur_lr = max(MIN_LR, lr_scheduler.get_last_lr()[0])
            for pg in optimizer.param_groups:
                pg["lr"] = cur_lr
        step += 1
        window_loss += step_loss_acc
        window_clips += step_clips_acc
        window_tokens += step_tokens_acc
        window_steps += 1
        step_loss_acc = 0.0
        step_clips_acc = 0
        step_tokens_acc = 0

    if stop_requested or step >= total_training_steps:
        break

elapsed_total = time.perf_counter() - run_start_time
run_summary = {"run_mode": RUN_MODE, "model_id": MODEL_ID, "planned_steps": total_training_steps, "completed_steps": step, "optimizer_updates": optimizer_updates, "elapsed_seconds": elapsed_total, "nonfinite_batches": nonfinite_batches, "skipped_batches": skipped_batches, "stop_requested": stop_requested, "metrics": metrics}
with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)
print(f"\n[TRAINING COMPLETE] {step}/{total_training_steps} optimizer steps completed in {elapsed_total/60:.1f} minutes.")
print(f"Metrics saved to {METRICS_CSV} and {METRICS_JSON}.")
print(f"Collator skip rate: {collator.skipped_samples}/{collator.total_samples} samples dropped ({collator.skipped_samples/max(1, collator.total_samples)*100:.2f}%).")


## 10. Post-Training FinTech Policy Gauntlet & Forgetting Audit

Evaluates:
1. **FinTech Policy Gauntlet**: Re-evaluates the 5 policy scenarios on the fine-tuned model and compares with pre-training baseline.
2. **Anti-Catastrophic Forgetting Audit**: Re-evaluates post-training accuracy on held-out FinQA / TAT-QA benchmarks.
3. **Paired Discordant Analysis**: Measures retained items, forgotten items, newly learned items, and persistent failures.


In [ ]:
print("=== Running Post-Tuning FinTech Invariant Gauntlet ===\n")
post_gauntlet_score, _ = run_fintech_gauntlet(model, tokenizer, eval_scenarios, title="Post-Training Model Gauntlet")

print("\n" + "=" * 60)
print(f"GAUNTLET COMPARATIVE SCORECARD:")
print(f"- Pre-Training Base Model:  {pre_gauntlet_score}/{len(eval_scenarios)} Passed")
print(f"- Post-Training Tuned Model: {post_gauntlet_score}/{len(eval_scenarios)} Passed")
print("=" * 60)

# Anti-Catastrophic Forgetting Audit
print("\n=== Running Anti-Catastrophic Forgetting Audit ===")
post_acc, post_lower, post_upper, post_correct_set = evaluate_anchor_accuracy(model, tokenizer, anchor_eval_suite)
retention_pct = (post_acc / max(pre_acc, 0.01)) * 100
degradation = max(0.0, (pre_acc - post_acc) * 100)

print(f"Pre-Training Anchor Accuracy:  {pre_acc * 100:.1f}% (95% Wilson CI: [{pre_lower * 100:.1f}%, {pre_upper * 100:.1f}%])")
print(f"Post-Training Anchor Accuracy: {post_acc * 100:.1f}% (95% Wilson CI: [{post_lower * 100:.1f}%, {post_upper * 100:.1f}%])")
print(f"Empirical Anchor Retention:    {retention_pct:.1f}%")
print(f"Catastrophic Degradation:      {degradation:.1f}%")

# Paired Discordant Analysis
retained = len(pre_correct_set & post_correct_set)
forgotten = len(pre_correct_set - post_correct_set)
newly_learned = len(post_correct_set - pre_correct_set)
both_failed = len(anchor_eval_suite) - len(pre_correct_set | post_correct_set)

print(f"\nPaired Item Analysis (N={len(anchor_eval_suite)}):")
print(f"- Retained Correct: {retained}/{len(anchor_eval_suite)}")
print(f"- Forgotten Items:  {forgotten}/{len(anchor_eval_suite)}")
print(f"- Newly Correct:    {newly_learned}/{len(anchor_eval_suite)}")
print(f"- Both Failed:      {both_failed}/{len(anchor_eval_suite)}")

evaluation_summary = {"model_id": MODEL_ID, "run_mode": RUN_MODE, "pre_anchor_accuracy": pre_acc, "post_anchor_accuracy": post_acc, "pre_anchor_wilson_ci": [pre_lower, pre_upper], "post_anchor_wilson_ci": [post_lower, post_upper], "anchor_retention_pct": retention_pct, "anchor_degradation_percentage_points": degradation, "pre_gauntlet_passed": pre_gauntlet_score, "post_gauntlet_passed": post_gauntlet_score, "gauntlet_total": len(eval_scenarios), "retained_correct": retained, "forgotten": forgotten, "newly_learned": newly_learned, "both_failed": both_failed}
with open(f"{OUTPUT_DIR}_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)
print(f"Evaluation summary saved to {OUTPUT_DIR}_evaluation.json")


## 11. Checkpoint Serialization & Export


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Successfully exported fine-tuned LoRA adapter to {OUTPUT_DIR}!")
print("Saved artifacts:")
for f in os.listdir(OUTPUT_DIR):
    p = os.path.join(OUTPUT_DIR, f)
    print(f"  - {f} ({os.path.getsize(p) / (1024**2):.2f} MB)")
